<a href="https://colab.research.google.com/github/RohanRathaur1/AAI2025/blob/main/capstone_project_for_118s.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
%pip install --upgrade --quiet langchain langchain-core langchain-openai langchain-community langgraph pypdf langchain-chroma openpyxl chromadb faiss-cpu

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
import os
from openai import OpenAI
from google.colab import userdata


api_key = userdata.get("OpenAi")
os.environ["OPENAI_API_KEY"] = api_key


model = ChatOpenAI(
    model="gpt-4o",
    temperature=0.7
)


embedding = OpenAIEmbeddings(
    model="text-embedding-3-large"
)

In [ ]:
import time
import faiss
import numpy as np
import os
from openai import OpenAI
from pypdf import PdfReader

client = OpenAI()

# AI Classifier Agent
def classify_issue_ai(description):
    """Categorizes the user's IT issue using the OpenAI model."""
    prompt = f"""
    You are an IT helpdesk classifier.
    Categorize the user's IT issue into ONE category from this list:

    - Account / Login Issue
    - Network Issue
    - Hardware / Performance Issue
    - Email Issue
    - Software Issue
    - Security Issue
    - Printer Issue
    - General IT Issue

    User description: "{description}"

    Respond ONLY with the category.
    """
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=20
    )
    return response.choices[0].message.content.strip()

# Knowledge Agent (RAG) using PDF Files
class KnowledgeAgent:
    """Handles PDF loading, text chunking, embedding, vector indexing (FAISS), and RAG search."""
    def __init__(self, pdf_paths):
        self.pdf_paths = pdf_paths
        self.pdf_chunks = {}
        self.pdf_embeddings = {}
        self.indexes = {}
        self.load_all_pdfs()

    def load_all_pdfs(self):
        """Loads and indexes all PDF files."""
        for path in self.pdf_paths:
            if not os.path.exists(path):
                print(f"Error: PDF file not found at {path}. Skipping.")
                continue

            pdf_name = os.path.basename(path)
            print(f"Loading and processing {pdf_name}...")
            text = self.extract_pdf_text(path)
            chunks = self.chunk_text(text)
            self.pdf_chunks[pdf_name] = chunks

            if not chunks:
                print(f"Warning: {pdf_name} is empty or extraction failed. Skipping indexing.")
                continue

            embeddings = self.embed(chunks)
            self.pdf_embeddings[pdf_name] = np.stack(embeddings)
            index = self.build_vector_index(embeddings)
            self.indexes[pdf_name] = index

    def extract_pdf_text(self, path):
        """Extracts all text from a PDF file."""
        reader = PdfReader(path)
        full_text = ""
        for page in reader.pages:
            t = page.extract_text()
            if t:
                full_text += t + "\n"
        return full_text

    def chunk_text(self, text, chunk_size=500):
        """Splits text into chunks of specified word size."""
        words = text.split()
        return [
            " ".join(words[i:i + chunk_size])
            for i in range(0, len(words), chunk_size)
        ]

    def embed(self, texts):
        """Generates embeddings for a list of text strings."""
        response = client.embeddings.create(
            model="text-embedding-3-large",
            input=texts
        )
        return [np.array(x.embedding, dtype="float32") for x in response.data]

    def build_vector_index(self, embeddings):
        """Builds a FAISS vector index."""
        dim = len(embeddings[0])
        index = faiss.IndexFlatL2(dim)
        index.add(np.stack(embeddings))
        return index

    def auto_select_pdf(self, query):
        """Finds the most relevant PDF based on the query."""
        query_vec = self.embed([query])[0].reshape(1, -1)
        best_pdf = None
        best_score = float("inf")
        for pdf_name, index in self.indexes.items():
            scores, indices = index.search(query_vec, 1)
            score = scores[0][0]
            if score < best_score:
                best_score = score
                best_pdf = pdf_name
        return best_pdf

    def search_pdf(self, query, pdf_name, k=3):
        """Searches a specific PDF for the top-k most relevant chunks."""
        query_vec = self.embed([query])[0].reshape(1, -1)
        index = self.indexes[pdf_name]
        chunks = self.pdf_chunks[pdf_name]
        scores, indices = index.search(query_vec, k)
        return [chunks[i] for i in indices[0]]

# Knowledge Agent Suggestion Agent
    def get_knowledge_suggestion(self, query, context):
        """Generates a concise, single-sentence preventative tip/suggestion."""
        prompt = f"""
        You are an IT helpdesk assistant providing a preventative tip.

        Based ONLY on the following PDF content, extract one single, concise, actionable
        suggestion or preventative tip for the user related to their issue.
        This tip should be a one-sentence instruction.

        PDF content:
        {context}

        User issue: "{query}"

        Respond ONLY with the one-sentence suggestion.
        """
        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=50
        )
        return response.choices[0].message.content.strip()

# Workflow Agent
    def get_workflow_and_suggestion(self, query):
        """Retrieves both the workflow step and the knowledge suggestion."""
        pdf_name = self.auto_select_pdf(query)

        if pdf_name is None:
            return "No automated steps available; escalate to human support.", "No relevant tip found."

        retrieved_chunks = self.search_pdf(query, pdf_name, k=5)
        context = "\n\n---\n".join(retrieved_chunks)

        workflow_prompt = f"""
        You are an IT workflow assistant.

        Using ONLY the following PDF content from {pdf_name}, provide the automatic next step for the user that is in the PDF file.
        Don't print: automated next step (user report), just whatever comes after that (without the quotations too).
        If there is no automated steps, say:
        "No automated steps available; escalate to human support."

        PDF content:
        {context}

        User issue: "{query}"
        """
        workflow_response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[{"role": "user", "content": workflow_prompt}],
            max_tokens=300
        )
        workflow_steps = workflow_response.choices[0].message.content.strip()

        suggestion = self.get_knowledge_suggestion(query, context)

        return workflow_steps, suggestion

#Escalation Agent
class EscalationAgent:
    """Simulates the process of escalating an issue."""
    def escalate_to_human(self, name, issue_description):
        print(f"[Escalation] Escalating issue for {name} to human IT support...")
        return f"Issue for {name} has been escalated to human support. Description: '{issue_description}'"

#  Workflow Agent
class WorkflowAgent:
    """Coordinates the retrieval and execution of workflow information."""
    def __init__(self, knowledge_agent):
        self.knowledge_agent = knowledge_agent

    def execute_workflow(self, issue_description):
        """Calls the KnowledgeAgent to get the action and the tip."""
        return self.knowledge_agent.get_workflow_and_suggestion(issue_description)

# Multi-Agent Pipeline
def intake_agent():
    """The main function that runs the entire agent pipeline."""
    print("--- 🤖 IT Support System 🤖 ---")
    name = input("Your name: ")
    department = input("Your department: ")
    issue_description = input("Describe your IT problem: ")
    print("\nProcessing request...")

    pdf_files = [
       "/content/Account Login 118.pdf",
        "/content/General IT 118.pdf",
        "/content/Hardware 118.pdf",
        "/content/Network Issue 118.pdf",
        "/content/Printer Issue 118.pdf",
        "/content/Security 118.pdf",
        "/content/Software 118.pdf"
    ]

    # 1. AI Classification
    category = classify_issue_ai(issue_description)

    # 2. Initialize KnowledgeAgent with PDFs
    kn_agent = KnowledgeAgent(pdf_files)

    # 3. Workflow automation using KnowledgeAgent
    wf_agent = WorkflowAgent(kn_agent)
    # Unpacks the automated action and the suggestion
    workflow_steps, knowledge_suggestion = wf_agent.execute_workflow(issue_description)

    # 4. If no workflow returned, escalate
    if "No automated steps" in workflow_steps:
        workflow_steps = EscalationAgent().escalate_to_human(name, issue_description)

    #  Summary Output
    print("\n--- Summary ---")
    print(f"Name: {name}")
    print(f"Department: {department}")
    print(f"Issue Description: {issue_description}")
    print(f"AI Category: {category}")

    if "No automated steps" not in workflow_steps:
        print("\n--- Knowledge Agent Proactive Tip <3 --- ")
        print(f"💡{knowledge_suggestion}")

    print("\n--- Workflow Agent Execution ---")
    print(workflow_steps)


if __name__ == "__main__":
    intake_agent()

--- 🤖 IT Support System 🤖 ---
Your name: Josue 
Your department: Marketing
Describe your IT problem: My computer won't turn on. I believe the battery is dead. 

Processing request...
Loading and processing Account Login 118.pdf...
Loading and processing General IT 118.pdf...
Loading and processing Hardware 118.pdf...
Loading and processing Network Issue 118.pdf...
Loading and processing Printer Issue 118.pdf...
Loading and processing Security 118.pdf...
Loading and processing Software 118.pdf...

--- Summary ---
Name: Josue 
Department: Marketing
Issue Description: My computer won't turn on. I believe the battery is dead. 
AI Category: Hardware / Performance Issue

--- Knowledge Agent Proactive Tip <3 --- 
💡Verify that the power cable is securely plugged into both your device and the wall outlet or UPS, and if using a laptop, check that the charger block light is on.

--- Workflow Agent Execution ---
A hardware failure has been detected on your device. Your incident has been escalated 